# Low-res simulation: Reverse-L column kernel vs Hybrid Lean

This notebook replaces the crashed real-data column fit with a single low-resolution simulation benchmark.

Grid: lat x4, lon x2 relative to the original GEMS grid.

Models fitted once:

- `ColumnReverseL_A4_R80_Lag2`: regular-grid reverse-L template kernel.
- `Hybrid_Lean_L08F04_C4F03_Op0p126`: existing hybrid kernel, total conditioning 41.


In [1]:
import os, sys, time, gc, io, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import orderings as _orderings
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit
from GEMS_TCO.kernels_vecchia_column import ReverseLColumnVecchiaFit

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
print("SRC:", SRC)
print("DEVICE:", DEVICE)
print("torch:", torch.__version__)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
DEVICE: cpu
torch: 2.5.1


In [2]:
# Low-resolution x4/x2 simulation config
DELTA_LAT = 0.044 * 4
DELTA_LON = 0.063 * 2
LAT_RANGE = (-3.0, 2.0)
LON_RANGE = (121.0, 131.0)
T_STEPS = 8
SEED = 123

SMOOTH = 0.5
MM_COND_NUMBER = 100
NHEADS = 0
DAILY_STRIDE = 2

# One-shot LBFGS settings
LBFGS_STEPS = 5
LBFGS_EVAL = 20
LBFGS_HIST = 10
INIT_NOISE = 0.25
SUPPRESS_FIT_PRINTS = False

TRUE_DICT = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "advec_lon": -0.16,
    "nugget": 2.5,
}

LEAN_OFFSET = DELTA_LON  # Op0p126, one low-res lon cell

COLUMN_SPEC = {
    "model": "ColumnReverseL_A4_R80_Lag2",
    "head_right_cols": 3,
    "above_count": 4,
    "right_col_count": 3,
    "right_neighbor_count": 80,
    "lag_count": 2,
    "include_lag_self": False,
}

HYBRID_SPEC = {
    "model": "Hybrid_Lean_L08F04_C4F03_Op0p126",
    "limit_A": 20,
    "lag1_local_count": 8,
    "lag1_fresh_count": 4,
    "lag2_local_count": 4,
    "lag2_fresh_count": 3,
    "pred_lag1_lon_offset": LEAN_OFFSET,
}

OUT_CSV = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/sim_vecchia_lowres_x4_lonx2_column_vs_lean_050526.csv')
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("resolution:", DELTA_LAT, DELTA_LON)
print("true:", TRUE_DICT)
print("hybrid offset:", LEAN_OFFSET)


resolution: 0.176 0.126
true: {'sigmasq': 10.0, 'range_lat': 0.3, 'range_lon': 0.4, 'range_time': 2.0, 'advec_lat': 0.08, 'advec_lon': -0.16, 'nugget': 2.5}
hybrid offset: 0.126


In [3]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
SPATIAL_KEYS = ["sigmasq", "range_lat", "range_lon"]
ADVECTION_KEYS = ["advec_lat", "advec_lon"]


def true_to_log_params(d):
    phi2 = 1.0 / d["range_lon"]
    phi1 = d["sigmasq"] * phi2
    phi3 = (d["range_lon"] / d["range_lat"]) ** 2
    phi4 = (d["range_lon"] / d["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d["advec_lat"], d["advec_lon"], np.log(d["nugget"])]


def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1]); phi3 = np.exp(p[2]); phi4 = np.exp(p[3]); rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }


def make_random_init(rng, true_log, init_noise):
    noisy = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        noisy[i] = true_log[i] + rng.uniform(-init_noise, init_noise)
    for i in [4, 5]:
        noisy[i] = true_log[i] + rng.uniform(-0.03, 0.03)
    return noisy


def rmsre_for_keys(est, truth, keys, zero_thresh=0.01):
    vals = []
    for k in keys:
        tv = truth[k]
        if abs(tv) >= zero_thresh:
            vals.append(((est[k] - tv) / abs(tv)) ** 2)
        else:
            vals.append(abs(est[k] - tv) ** 2)
    return float(np.sqrt(np.mean(vals)))


def calculate_metrics(out_params, truth):
    est = backmap_params(out_params)
    return {
        "overall_rmsre": rmsre_for_keys(est, truth, P_LABELS),
        "spatial_rmsre": rmsre_for_keys(est, truth, SPATIAL_KEYS),
        "advec_rmsre": rmsre_for_keys(est, truth, ADVECTION_KEYS),
        "range_time_re": abs(est["range_time"] - truth["range_time"]) / abs(truth["range_time"]),
        "nugget_re": abs(est["nugget"] - truth["nugget"]) / abs(truth["nugget"]),
        "est": est,
    }


In [4]:
def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)


def build_target_grid(lat_range, lon_range):
    lat0, lat1 = float(min(lat_range)), float(max(lat_range))
    lon0, lon1 = float(min(lon_range)), float(max(lon_range))
    n_lat = int(np.floor((lat1 - lat0) / DELTA_LAT + 1e-9)) + 1
    n_lon = int(np.floor((lon1 - lon0) / DELTA_LON + 1e-9)) + 1
    lats = lat0 + torch.arange(n_lat, device=DEVICE, dtype=DTYPE) * DELTA_LAT
    lons = lon0 + torch.arange(n_lon, device=DEVICE, dtype=DTYPE) * DELTA_LON
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    return lats, lons, grid_coords


def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps
    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT
    lx[px // 2:] -= px * DELTA_LAT
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON
    ly[py // 2:] -= py * DELTA_LON
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params.cpu().float())
    S = torch.fft.fftn(C)
    S.real = torch.clamp(S.real, min=0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S.real) * noise).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)


def assemble_reg_map(field, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    reg_map = {}
    for t_idx in range(field.shape[-1]):
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.zeros((n_grid, 11), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = float(t_offset + t_idx)
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        reg_map[f"t{t_idx}"] = rows.detach()
    return reg_map


def compute_grid_ordering(grid_coords, mm_cond_number):
    coords_np = grid_coords.detach().cpu().numpy()
    ord_grid = _orderings.maxmin_cpp(coords_np)
    nns_grid = _orderings.find_nns_l2(locs=coords_np[ord_grid], max_nn=mm_cond_number)
    return ord_grid, nns_grid


In [5]:
def fit_column(reg_map_ord, ordered_grid_coords_np, initial_vals, truth):
    params = [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in initial_vals]
    model = ReverseLColumnVecchiaFit(
        smooth=SMOOTH,
        input_map=reg_map_ord,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=ordered_grid_coords_np,
        head_right_cols=COLUMN_SPEC["head_right_cols"],
        above_count=COLUMN_SPEC["above_count"],
        right_col_count=COLUMN_SPEC["right_col_count"],
        right_neighbor_count=COLUMN_SPEC["right_neighbor_count"],
        lag_count=COLUMN_SPEC["lag_count"],
        include_lag_self=COLUMN_SPEC["include_lag_self"],
        target_chunk_size=2048 if DEVICE.type == "cpu" else 4096,
    )

    t0 = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t0

    optimizer = model.set_optimizer(params, lr=1.0, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t1 = time.time()
    out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t1

    metrics = calculate_metrics(out, truth)
    row = {
        "model": COLUMN_SPEC["model"],
        "kernel": "column_reverse_l",
        "total_conditioning_nominal": (COLUMN_SPEC["above_count"] + COLUMN_SPEC["right_neighbor_count"]) * (COLUMN_SPEC["lag_count"] + 1),
        "n_heads": int(model.Heads_data.shape[0]),
        "n_tails": int(model.n_tails),
        "n_templates": int(len(model.Grouped_Batches)),
        "loss": float(out[-1]),
        "fit_iter": int(n_iter),
        "precompute_s": pre_s,
        "fit_s": fit_s,
        "total_s": pre_s + fit_s,
        **{k: v for k, v in metrics.items() if k != "est"},
        **{f"est_{k}": v for k, v in metrics["est"].items()},
    }
    return row


def fit_hybrid_lean(reg_map_ord, nns_grid, ordered_grid_coords_np, initial_vals, truth):
    params = [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in initial_vals]
    model = HybridVecchiaFit(
        smooth=SMOOTH,
        input_map=reg_map_ord,
        nns_map=nns_grid,
        mm_cond_number=MM_COND_NUMBER,
        nheads=NHEADS,
        limit_A=HYBRID_SPEC["limit_A"],
        limit_B_local=HYBRID_SPEC["lag1_local_count"],
        limit_C_local=HYBRID_SPEC["lag2_local_count"],
        daily_stride=DAILY_STRIDE,
        spatial_coords=ordered_grid_coords_np,
        lag1_lon_offset=HYBRID_SPEC["pred_lag1_lon_offset"],
        lag1_fresh_count=HYBRID_SPEC["lag1_fresh_count"],
        lag2_fresh_count=HYBRID_SPEC["lag2_fresh_count"],
    )

    t0 = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t0

    optimizer = model.set_optimizer(params, lr=1.0, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t1 = time.time()
    out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t1

    metrics = calculate_metrics(out, truth)
    row = {
        "model": HYBRID_SPEC["model"],
        "kernel": "hybrid_lean",
        "total_conditioning_nominal": 20 + (1 + 8 + 4) + (1 + 4 + 3),
        "n_heads": 0,
        "n_tails": int(model.n_tails),
        "n_templates": np.nan,
        "loss": float(out[-1]),
        "fit_iter": int(n_iter),
        "precompute_s": pre_s,
        "fit_s": fit_s,
        "total_s": pre_s + fit_s,
        **{k: v for k, v in metrics.items() if k != "est"},
        **{f"est_{k}": v for k, v in metrics["est"].items()},
    }
    return row


In [6]:
# Build one simulated dataset
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

lats_grid, lons_grid, grid_coords = build_target_grid(LAT_RANGE, LON_RANGE)
n_lat, n_lon = len(lats_grid), len(lons_grid)
print(f"Grid: {n_lat} x {n_lon} x {T_STEPS} = {n_lat * n_lon * T_STEPS:,} rows")

ord_grid, nns_grid = compute_grid_ordering(grid_coords, MM_COND_NUMBER)
ordered_grid_coords_np = grid_coords[ord_grid].detach().cpu().numpy()
print("Maxmin ordering done")

true_log = true_to_log_params(TRUE_DICT)
true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)
initial_vals = make_random_init(rng, true_log, INIT_NOISE)
print("Initial vals:", [round(x, 4) for x in initial_vals])

field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
reg_map = assemble_reg_map(field, grid_coords, true_params)
del field
reg_map_ord = {k: v[ord_grid].contiguous() for k, v in reg_map.items()}

print("data ready")


Grid: 29 x 80 x 8 = 18,560 rows
Maxmin ordering done
Initial vals: [3.3101, 0.6932, 0.4355, -3.3767, 0.0987, -0.1346, 0.7542]
data ready


In [7]:
rows = []
for fit_name in ["column", "hybrid"]:
    print("\n" + "=" * 80)
    print("Fitting", fit_name)
    if fit_name == "column":
        row = fit_column(reg_map_ord, ordered_grid_coords_np, initial_vals, TRUE_DICT)
    else:
        row = fit_hybrid_lean(reg_map_ord, nns_grid, ordered_grid_coords_np, initial_vals, TRUE_DICT)
    row.update({f"true_{k}": v for k, v in TRUE_DICT.items()})
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    print(row)
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df



Fitting column
Pre-computing ReverseLColumnVecchia [heads_right=3, above=4, right_cols=3, right_n=80, lags=2]... Done in 4.5s. grid=29x80, heads=696, tails=17864, templates=81, m mean/med/max=219.6/252/252
--- Starting Reverse-L Column Vecchia L-BFGS ---
--- Step 1/5 / Loss: 1.459091 / Max Grad: 7.84e-05 ---
--- Step 2/5 / Loss: 1.455969 / Max Grad: 1.09e-05 ---
--- Step 3/5 / Loss: 1.455969 / Max Grad: 1.09e-05 ---
--- Step 4/5 / Loss: 1.455969 / Max Grad: 1.09e-05 ---
--- Step 5/5 / Loss: 1.455969 / Max Grad: 1.09e-05 ---
{'model': 'ColumnReverseL_A4_R80_Lag2', 'kernel': 'column_reverse_l', 'total_conditioning_nominal': 252, 'n_heads': 696, 'n_tails': 17864, 'n_templates': 81, 'loss': 1.4559691921394162, 'fit_iter': 4, 'precompute_s': 4.557503938674927, 'fit_s': 16.16756796836853, 'total_s': 20.725071907043457, 'overall_rmsre': 0.055821859767104326, 'spatial_rmsre': 0.044297958232002874, 'advec_rmsre': 0.08465119152514315, 'range_time_re': 0.030912335527605528, 'nugget_re': 0.025266

,model,kernel,total_conditioning_nominal,n_heads,n_tails,n_templates,loss,fit_iter,precompute_s,fit_s,...,est_advec_lat,est_advec_lon,est_nugget,true_sigmasq,true_range_lat,true_range_lon,true_range_time,true_advec_lat,true_advec_lon,true_nugget
0,ColumnReverseL_A4_R80_Lag2,column_reverse_l,252,696,17864,81.0,1.455969,4,4.557504,16.167568,...,0.070821,-0.165462,2.436833,10.0,0.3,0.4,2.0,0.08,-0.16,2.5
1,Hybrid_Lean_L08F04_C4F03_Op0p126,hybrid_lean,41,0,18560,NaN,1.565021,1,1.225008,28.617854,...,0.070197,-0.150324,2.581242,10.0,0.3,0.4,2.0,0.08,-0.16,2.5


In [8]:
display_cols = [
    "model", "kernel", "total_conditioning_nominal", "n_heads", "n_tails", "n_templates",
    "loss", "overall_rmsre", "spatial_rmsre", "advec_rmsre", "range_time_re", "nugget_re",
    "precompute_s", "fit_s", "total_s",
    "est_range_lat", "est_range_lon", "est_range_time", "est_advec_lat", "est_advec_lon", "est_nugget",
]
print("Saved:", OUT_CSV)
display(df[display_cols])


Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/sim_vecchia_lowres_x4_lonx2_column_vs_lean_050526.csv


,model,kernel,total_conditioning_nominal,n_heads,n_tails,n_templates,loss,overall_rmsre,spatial_rmsre,advec_rmsre,...,nugget_re,precompute_s,fit_s,total_s,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget
0,ColumnReverseL_A4_R80_Lag2,column_reverse_l,252,696,17864,81.0,1.455969,0.055822,0.044298,0.084651,...,0.025267,4.557504,16.167568,20.725072,0.285759,0.379693,1.938175,0.070821,-0.165462,2.436833
1,Hybrid_Lean_L08F04_C4F03_Op0p126,hybrid_lean,41,0,18560,NaN,1.565021,0.084876,0.082915,0.096620,...,0.032497,1.225008,28.617854,29.842862,0.327724,0.427625,2.200759,0.070197,-0.150324,2.581242


## Reading guide

This is a smoke benchmark, not a final MC result.

What to check first:

- `n_templates`: if this is small, reverse-L template reuse is actually active.
- `total_s`: compare wall time against the Lean hybrid.
- `loss` and RMSRE: only one simulated dataset, so do not overinterpret.

If the column kernel is still unstable here, the issue is likely implementation-level rather than real-data memory pressure.
